In [1]:
import sys

print(sys.executable)

C:\Users\kevin\Finding Job\Project\airline-ops-platform\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import sys

import pandas as pd

data_dir = Path("data/raw/bts").resolve()

print("Python：", sys.executable)
print("資料夾：", data_dir)
print("資料夾存在：", data_dir.exists())
print("資料夾內容：", list(data_dir.iterdir()))

Python： C:\Users\kevin\Finding Job\Project\airline-ops-platform\.venv\Scripts\python.exe
資料夾： C:\Users\kevin\Finding Job\Project\airline-ops-platform\data\raw\bts
資料夾存在： True
資料夾內容： [WindowsPath('C:/Users/kevin/Finding Job/Project/airline-ops-platform/data/raw/bts/.gitkeep'), WindowsPath('C:/Users/kevin/Finding Job/Project/airline-ops-platform/data/raw/bts/T_ONTIME_REPORTING.csv')]


In [3]:
data_files = [
    path
    for path in data_dir.iterdir()
    if path.suffix.lower() in {".csv", ".zip"}
]

data_files

[WindowsPath('C:/Users/kevin/Finding Job/Project/airline-ops-platform/data/raw/bts/T_ONTIME_REPORTING.csv')]

In [4]:
data_path = data_files[0]

sample = pd.read_csv(
    data_path,
    nrows=10_000,
    low_memory=False,
)

In [5]:
candidate_columns = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "TAIL_NUM",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
]

sample[candidate_columns].head(20)

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,TAIL_NUM,ORIGIN,DEST,CRS_DEP_TIME
0,1/1/2026 0:00,AA,1480,N101NN,LAX,BOS,1358
1,1/1/2026 0:00,AA,255,N101NN,JFK,LAX,941
2,1/1/2026 0:00,AA,2453,N103NN,LAX,BOS,2200
3,1/1/2026 0:00,AA,302,N103NN,JFK,LAX,1745
4,1/1/2026 0:00,AA,307,N103NN,LAX,JFK,830
5,1/1/2026 0:00,AA,1479,N103US,MKE,CLT,710
6,1/1/2026 0:00,AA,1775,N103US,CLT,DCA,1902
7,1/1/2026 0:00,AA,3005,N103US,CLT,OMA,1120
8,1/1/2026 0:00,AA,3005,N103US,OMA,CLT,1358
9,1/1/2026 0:00,AA,3085,N104NN,JFK,SNA,925


In [6]:
sample["OP_CARRIER_FL_NUM"].value_counts().head(10)

OP_CARRIER_FL_NUM
2301    7
1455    7
2476    6
891     6
767     6
1419    6
1123    6
419     6
2120    6
899     6
Name: count, dtype: int64

In [7]:
example_flight_number = (
    sample["OP_CARRIER_FL_NUM"]
    .value_counts()
    .index[0]
)

example_flight_number

np.int64(2301)

In [8]:
sample.loc[
    sample["OP_CARRIER_FL_NUM"] == example_flight_number,
    candidate_columns,
].head(30)

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,TAIL_NUM,ORIGIN,DEST,CRS_DEP_TIME
1163,1/1/2026 0:00,AA,2301,N673AW,BUF,CLT,1915
1164,1/1/2026 0:00,AA,2301,N673AW,CLT,BUF,1648
3753,1/1/2026 0:00,B6,2301,N559JB,JFK,FLL,859
5609,1/1/2026 0:00,DL,2301,N824DN,MSP,TPA,705
5610,1/1/2026 0:00,DL,2301,N824DN,TPA,MSP,1242
6820,1/1/2026 0:00,G4,2301,195NV,FAR,AZA,1902
7972,1/1/2026 0:00,NK,2301,N614NK,BUR,LAS,908


## Data Grain and Candidate Business Key

One row represents one scheduled flight leg operated by a carrier on a specific date.

Candidate business key:

- FL_DATE
- OP_UNIQUE_CARRIER
- OP_CARRIER_FL_NUM
- ORIGIN
- DEST

This key produced zero duplicate rows in the January 2026 dataset.

TAIL_NUM was excluded because aircraft assignments may change and can be null.
CRS_DEP_TIME was not required for uniqueness in the tested month.

In [9]:
columns_to_test = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "TAIL_NUM",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
]

key_data = pd.read_csv(
    data_path,
    usecols=columns_to_test,
    low_memory=False,
)

key_data.shape

(544003, 7)

In [10]:
candidate_key_v2 = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "DEST",
]

key_data[candidate_key_v2].isna().sum()

FL_DATE              0
OP_UNIQUE_CARRIER    0
OP_CARRIER_FL_NUM    0
ORIGIN               0
DEST                 0
dtype: int64

In [11]:
duplicate_mask_v2 = key_data.duplicated(
    subset=candidate_key_v2,
    keep=False,
)

duplicate_mask_v2.sum()

np.int64(0)

In [12]:
key_data.loc[
    duplicate_mask_v2,
    columns_to_test,
].sort_values(candidate_key_v2).head(30)

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,TAIL_NUM,ORIGIN,DEST,CRS_DEP_TIME


In [13]:
#null values and flight status discovery
status_columns = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY_NEW",
    "CRS_ARR_TIME",
    "ARR_TIME",
    "ARR_DELAY_NEW",
    "CANCELLED",
    "CANCELLATION_CODE",
    "DIVERTED",
    "ACTUAL_ELAPSED_TIME",
    "AIR_TIME",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
]

flight_status = pd.read_csv(
    data_path,
    usecols=status_columns,
    low_memory=False,
)

flight_status.shape

(544003, 19)

In [14]:
null_profile = pd.DataFrame({
    "null_count": flight_status.isna().sum(),
    "null_percent": (
        flight_status.isna().mean() * 100
    ).round(2),
}).sort_values("null_percent", ascending=False)

null_profile

,null_count,null_percent
CANCELLATION_CODE,518368,95.29
NAS_DELAY,436528,80.24
SECURITY_DELAY,436528,80.24
WEATHER_DELAY,436528,80.24
CARRIER_DELAY,436528,80.24
LATE_AIRCRAFT_DELAY,436528,80.24
ACTUAL_ELAPSED_TIME,26781,4.92
AIR_TIME,26781,4.92
ARR_DELAY_NEW,26781,4.92
ARR_TIME,25838,4.75


In [15]:
flight_status["CANCELLED"].value_counts(dropna=False)

CANCELLED
0    518368
1     25635
Name: count, dtype: int64

In [16]:
flight_status["DIVERTED"].value_counts(dropna=False)

DIVERTED
0    542857
1      1146
Name: count, dtype: int64

In [17]:
flight_status.loc[
    flight_status["CANCELLED"] == 1
].head(10)

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY_NEW,CRS_ARR_TIME,ARR_TIME,ARR_DELAY_NEW,CANCELLED,CANCELLATION_CODE,DIVERTED,ACTUAL_ELAPSED_TIME,AIR_TIME,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
1639,1/1/2026 0:00,AA,2603,735,NaN,NaN,1035,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1674,1/1/2026 0:00,AA,2374,1153,NaN,NaN,1353,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2444,1/1/2026 0:00,AA,756,904,NaN,NaN,1223,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2607,1/1/2026 0:00,AS,811,720,NaN,NaN,1119,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2608,1/1/2026 0:00,AS,876,1252,NaN,NaN,2033,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2670,1/1/2026 0:00,AS,207,2125,NaN,NaN,2,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2694,1/1/2026 0:00,AS,1341,705,NaN,NaN,922,NaN,NaN,1,B,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3122,1/1/2026 0:00,AS,748,740,NaN,NaN,1045,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3151,1/1/2026 0:00,AS,570,2217,NaN,NaN,6,NaN,NaN,1,B,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3431,1/1/2026 0:00,B6,601,550,NaN,NaN,856,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
cancelled_mask = flight_status["CANCELLED"] == 1
operated_mask = flight_status["CANCELLED"] == 0

cancellation_checks = pd.Series({
    "cancelled_rows": cancelled_mask.sum(),

    "cancelled_missing_code": (
        cancelled_mask
        & flight_status["CANCELLATION_CODE"].isna()
    ).sum(),

    "operated_with_cancellation_code": (
        operated_mask
        & flight_status["CANCELLATION_CODE"].notna()
    ).sum(),

    "cancelled_with_dep_time": (
        cancelled_mask
        & flight_status["DEP_TIME"].notna()
    ).sum(),

    "cancelled_with_arr_time": (
        cancelled_mask
        & flight_status["ARR_TIME"].notna()
    ).sum(),

    "cancelled_with_air_time": (
        cancelled_mask
        & flight_status["AIR_TIME"].notna()
    ).sum(),
})

cancellation_checks

cancelled_rows                     25635
cancelled_missing_code                 0
operated_with_cancellation_code        0
cancelled_with_dep_time              471
cancelled_with_arr_time                0
cancelled_with_air_time                0
dtype: int64

In [19]:
cancelled_with_dep_time = flight_status.loc[
    cancelled_mask
    & flight_status["DEP_TIME"].notna()
]

cancelled_with_dep_time[
    [
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "OP_CARRIER_FL_NUM",
        "CRS_DEP_TIME",
        "DEP_TIME",
        "DEP_DELAY_NEW",
        "CANCELLATION_CODE",
        "ARR_TIME",
        "AIR_TIME",
    ]
].head(20)

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY_NEW,CANCELLATION_CODE,ARR_TIME,AIR_TIME
9237,1/1/2026 0:00,OO,3310,820,929.0,NaN,B,NaN,NaN
9299,1/1/2026 0:00,OO,3060,820,817.0,0.0,B,NaN,NaN
10812,1/1/2026 0:00,OO,4643,921,937.0,NaN,B,NaN,NaN
14060,1/1/2026 0:00,WN,540,600,605.0,5.0,A,NaN,NaN
15021,1/1/2026 0:00,WN,3903,825,821.0,0.0,B,NaN,NaN
16303,1/1/2026 0:00,WN,4229,755,905.0,70.0,B,NaN,NaN
19692,1/2/2026 0:00,AA,1568,1210,906.0,1256.0,A,NaN,NaN
20899,1/2/2026 0:00,AS,186,2130,2147.0,17.0,C,NaN,NaN
21486,1/2/2026 0:00,B6,1138,1140,1135.0,0.0,A,NaN,NaN
21723,1/2/2026 0:00,B6,403,2155,155.0,240.0,C,NaN,NaN


In [20]:
cancelled_with_dep_time[
    "CANCELLATION_CODE"
].value_counts(dropna=False)

CANCELLATION_CODE
B    354
A     87
C     29
D      1
Name: count, dtype: int64

In [21]:
all_cancelled_distribution = (
    flight_status.loc[
        cancelled_mask,
        "CANCELLATION_CODE",
    ]
    .value_counts()
    .to_frame("count")
)

all_cancelled_distribution["percent"] = (
    all_cancelled_distribution["count"]
    / all_cancelled_distribution["count"].sum()
    * 100
).round(2)

all_cancelled_distribution

,count,percent
CANCELLATION_CODE,,
B,22463,87.63
A,2246,8.76
C,879,3.43
D,47,0.18


In [22]:
valid_cancellation_codes = {"A", "B", "C", "D"}

assert flight_status["CANCELLED"].isin([0, 1]).all()

assert flight_status.loc[
    cancelled_mask,
    "CANCELLATION_CODE",
].notna().all()

assert flight_status.loc[
    operated_mask,
    "CANCELLATION_CODE",
].isna().all()

assert flight_status[
    "CANCELLATION_CODE"
].dropna().isin(valid_cancellation_codes).all()

print("All cancellation data quality checks passed.")

All cancellation data quality checks passed.


In [23]:
delay_cause_columns = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
]

qualifying_delay_mask = (
    (flight_status["CANCELLED"] == 0)
    & (flight_status["DIVERTED"] == 0)
    & (flight_status["ARR_DELAY_NEW"] >= 15)
)

In [24]:
delay_checks = pd.Series({
    "qualifying_delay_rows": qualifying_delay_mask.sum(),

    "delayed_but_all_causes_null": (
        qualifying_delay_mask
        & flight_status[delay_cause_columns].isna().all(axis=1)
    ).sum(),

    "not_qualifying_but_has_cause": (
        ~qualifying_delay_mask
        & flight_status[delay_cause_columns].notna().any(axis=1)
    ).sum(),
})

delay_checks

qualifying_delay_rows           107475
delayed_but_all_causes_null          0
not_qualifying_but_has_cause         0
dtype: int64

## Data Quality Summary

Dataset scope: January 2026 BTS Reporting Carrier On-Time Performance.

### Core validation results

1. The candidate business key produced zero duplicate rows:
   - FL_DATE
   - OP_UNIQUE_CARRIER
   - OP_CARRIER_FL_NUM
   - ORIGIN
   - DEST

2. Cancellation data was consistent:
   - Cancelled flights had a valid cancellation code.
   - Operated flights had no cancellation code.
   - Some cancelled flights had DEP_TIME because they left the gate before cancellation.

3. Delay cause data was consistent:
   - Qualifying delays had at least one delay cause.
   - Non-qualifying flights had no delay cause values.

These findings are validated only for the January 2026 dataset and must be rechecked when additional months are ingested.

In [25]:
source_columns = pd.read_csv(
    data_path,
    nrows=0,
).columns.tolist()

flight_dates = pd.to_datetime(
    key_data["FL_DATE"],
    errors="coerce",
)

source_profile = pd.Series({
    "file_name": data_path.name,
    "file_size_mb": round(
        data_path.stat().st_size / 1024**2,
        2,
    ),
    "row_count": len(key_data),
    "column_count": len(source_columns),
    "minimum_flight_date": flight_dates.min(),
    "maximum_flight_date": flight_dates.max(),
    "invalid_flight_dates": flight_dates.isna().sum(),
})

source_profile

file_name               T_ONTIME_REPORTING.csv
file_size_mb                             91.13
row_count                               544003
column_count                                39
minimum_flight_date        2026-01-01 00:00:00
maximum_flight_date        2026-01-31 00:00:00
invalid_flight_dates                         0
dtype: object